In [1]:
import pandas as pd
import numpy as np

# 1. Configurando a aleatoriedade para ser idêntica
np.random.seed(2026)

# 2. Criando o esqueleto dos dados (12 meses de vendas)
meses = pd.date_range(start="2025-01-01", end="2025-12-01", freq="MS")
produtos = ["Action Figure Geek", "Escultura de Biscuit Luxo", "Estátua Colecionável 3D"]

data_linhas = []

# 3. Gerando os dados com Sazonalidade (Vendas sobem muito no fim do ano!)
for mes in meses:
    # Se for Novembro (Black Friday) ou Dezembro (Natal), o peso das vendas é maior
    if mes.month in [11, 12]:
        fator_sazonal = 2.5
    else:
        fator_sazonal = 1.0
        
    for produto in produtos:
        # Gerando uma quantidade de vendas base para cada produto
        vendas_base = np.random.randint(50, 150)
        quantidade_vendida = int(vendas_base * fator_sazonal)
        
        data_linhas.append({
            "Data_Mes": mes.strftime("%Y-%m"),
            "Produto": produto,
            "Quantidade_Vendida": quantidade_vendida
        })

# 4. Transformando em DataFrame e salvando o CSV limpinho
df_demanda = pd.DataFrame(data_linhas)
df_demanda.to_csv("demanda_colecionaveis.csv", index=False)

print(df_demanda.head(6))

  Data_Mes                    Produto  Quantidade_Vendida
0  2025-01         Action Figure Geek                  51
1  2025-01  Escultura de Biscuit Luxo                  56
2  2025-01    Estátua Colecionável 3D                  76
3  2025-02         Action Figure Geek                 106
4  2025-02  Escultura de Biscuit Luxo                 127
5  2025-02    Estátua Colecionável 3D                 127


In [4]:
import pandas as pd

# 1. Carregar o arquivo de vendas
df_demanda = pd.read_csv("demanda_colecionaveis.csv")

# 2. Agrupar as vendas por mês (Evolução Mensal)
vendas_por_mes = df_demanda.groupby("Data_Mes")["Quantidade_Vendida"].sum().reset_index()

# 3. Criar a coluna de Previsão usando a Média Móvel de 2 meses
vendas_por_mes['Previsao_Proximo_Mes'] = vendas_por_mes['Quantidade_Vendida'].rolling(window=2, min_periods=1).mean()

# 4. Calcular o erro absoluto de cada mês (Realidade - Previsão)
vendas_por_mes['Erro_Absoluto'] = (vendas_por_mes['Quantidade_Vendida'] - vendas_por_mes['Previsao_Proximo_Mes']).abs()

# 5. Calcular o MAD (Média dos Erros Absolutos, ignorando Janeiro no índice 0)
mad_final = vendas_por_mes['Erro_Absoluto'].iloc[1:].mean()

# ==========================================
# PRINT DOS RESULTADOS NA TELA
# ==========================================

# 6. Ranking Anual por Produto
vendas_por_produto = df_demanda.groupby("Produto")["Quantidade_Vendida"].sum().reset_index()
vendas_por_produto = vendas_por_produto.sort_values(by="Quantidade_Vendida", ascending=False)

print("--- 🏆 RANKING DE VENDAS POR PRODUTO (ANUAL) ---")
print(vendas_por_produto.to_string(index=False))
print("\n" + "="*55 + "\n")

print("--- TABELA DE PLANEJAMENTO DE DEMANDA COM ERRO (MAD) ---")
print(vendas_por_mes[['Data_Mes', 'Quantidade_Vendida', 'Previsao_Proximo_Mes', 'Erro_Absoluto']].to_string(index=False))
print("\n" + "="*55)
print(f"O ERRO MÉDIO (MAD) DO NOSSO MODELO É DE: {mad_final:.1f} UNIDADES")
print("="*55)

--- 🏆 RANKING DE VENDAS POR PRODUTO (ANUAL) ---
                  Produto  Quantidade_Vendida
Escultura de Biscuit Luxo                1587
       Action Figure Geek                1539
  Estátua Colecionável 3D                1406


--- TABELA DE PLANEJAMENTO DE DEMANDA COM ERRO (MAD) ---
Data_Mes  Quantidade_Vendida  Previsao_Proximo_Mes  Erro_Absoluto
 2025-01                 183                 183.0            0.0
 2025-02                 360                 271.5           88.5
 2025-03                 258                 309.0           51.0
 2025-04                 323                 290.5           32.5
 2025-05                 283                 303.0           20.0
 2025-06                 280                 281.5            1.5
 2025-07                 273                 276.5            3.5
 2025-08                 269                 271.0            2.0
 2025-09                 303                 286.0           17.0
 2025-10                 287                 295.